In [1]:
%pwd

'd:\\Siam\\KIdney disease\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\KIdney disease'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    dataset_dir: Path
    model_dir: Path
    model_name: str
    save_best_only: bool
    monitor_metric: str
    early_stopping: bool
    patience: int


In [6]:
from kidney_disease_classification.constants import *
from kidney_disease_classification.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])

    def get_training_config(self) -> TrainingConfig:
        config = self.config["training"]

        create_directories([config["root_dir"], config["model_dir"]])

        training_config = TrainingConfig(
            root_dir=config["root_dir"],
            dataset_dir=config["dataset_dir"],
            model_dir=config["model_dir"],
            model_name=config["model_name"],
            save_best_only=config["save_best_only"],
            monitor_metric=config["monitor_metric"],
            early_stopping=config["early_stopping"],
            patience=config["patience"]
        )
        return training_config


In [8]:
import os
import sys
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path

from kidney_disease_classification.exception import CustomException
from kidney_disease_classification.logger import logging

In [ ]:
class Training:
    def __init__(self, config: TrainingConfig, params):
        self.config = config
        self.params = params

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------------------
    # 1. DATA LOADERS (TRAIN + VAL ONLY)
    # ---------------------------
    def get_data_loaders(self):
        try:
            logging.info("Loading train and validation datasets...")

            img_size = tuple(self.params["IMAGE_SIZE"])

            # Train transform (augmentation only here)
            train_transform = transforms.Compose([
                transforms.Resize(img_size),
                transforms.Grayscale(num_output_channels=3),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5])
            ])

            # Validation transform (NO augmentation)
            val_transform = transforms.Compose([
                transforms.Resize(img_size),
                transforms.Grayscale(num_output_channels=3),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5])
            ])

            train_dir = Path(self.config.dataset_dir) / "train"
            val_dir = Path(self.config.dataset_dir) / "val"

            train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
            val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

            train_loader = DataLoader(
                train_dataset,
                batch_size=self.params["BATCH_SIZE"],
                shuffle=True
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=self.params["BATCH_SIZE"],
                shuffle=False
            )

            logging.info("Data loaders ready")

            return train_loader, val_loader

        except Exception as e:
            raise CustomException(e, sys)

    # ---------------------------
    # 2. MODEL BUILDING
    # ---------------------------
    def build_model(self):
        try:
            logging.info("Building EfficientNet model...")

            model = models.efficientnet_b0(pretrained=True)

            # Freeze backbone
            for param in model.features.parameters():
                param.requires_grad = False

            # Replace classifier
            in_features = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(in_features, self.params["NUM_CLASSES"])

            return model.to(self.device)

        except Exception as e:
            raise CustomException(e, sys)

    # ---------------------------
    # 3. TRAIN ONE EPOCH
    # ---------------------------
    def train_one_epoch(self, model, loader, criterion, optimizer):
        model.train()

        total_loss = 0

        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device).float()

            optimizer.zero_grad()

            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        return total_loss / len(loader)

    # ---------------------------
    # 4. VALIDATION (ONLY FOR MONITORING)
    # ---------------------------
    def validate(self, model, loader, criterion):
        model.eval()

        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device)
                labels = labels.to(self.device).float()

                outputs = model(images).squeeze()
                loss = criterion(outputs, labels)

                total_loss += loss.item()

                preds = torch.sigmoid(outputs) > 0.5
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        accuracy = correct / total
        return total_loss / len(loader), accuracy

    # ---------------------------
    # 5. TRAINING PIPELINE
    # ---------------------------
    def initiate_training(self):
        try:
            logging.info("Training stage started")

            train_loader, val_loader = self.get_data_loaders()
            model = self.build_model()

            criterion = nn.BCEWithLogitsLoss()

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=self.params["LEARNING_RATE"],
                weight_decay=self.params["WEIGHT_DECAY"]
            )

            best_val_acc = 0.0

            for epoch in range(self.params["EPOCHS"]):
                train_loss = self.train_one_epoch(model, train_loader, criterion, optimizer)
                val_loss, val_acc = self.validate(model, val_loader, criterion)

                logging.info(
                    f"Epoch [{epoch+1}/{self.params['EPOCHS']}] "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Val Loss: {val_loss:.4f} | "
                    f"Val Acc: {val_acc:.4f}"
                )

                # Save best model
                if val_acc > best_val_acc:
                    best_val_acc = val_acc

                    model_path = os.path.join(self.config.model_dir, self.config.model_name)

                    torch.save(model.state_dict(), model_path)

                    logging.info(f"Best model saved with accuracy: {best_val_acc:.4f}")

            logging.info("Training stage completed successfully")

            return True

        except Exception as e:
            raise CustomException(e, sys)

In [10]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config, params=config.params)
    training.initiate_training()
    
except Exception as e:
    raise CustomException(e, sys)


[2026-05-14 01:05:34,135: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-14 01:05:34,139: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-14 01:05:34,140: INFO: common: created directory at: artifacts]
[2026-05-14 01:05:34,142: INFO: common: created directory at: artifacts\training]
[2026-05-14 01:05:34,144: INFO: 293278742: Training stage started]
[2026-05-14 01:05:34,145: INFO: 293278742: Loading train and validation datasets...]
[2026-05-14 01:05:34,148: INFO: 293278742: Data loaders ready]
[2026-05-14 01:05:34,149: INFO: 293278742: Building EfficientNet model...]


d:\Siam\KIdney disease\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Siam\KIdney disease\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[2026-05-14 01:05:50,097: INFO: 293278742: Epoch [1/2] Train Loss: 0.6861 | Val Loss: 0.6977 | Val Acc: 0.5652]
[2026-05-14 01:05:50,140: INFO: 293278742: Best model saved with accuracy: 0.5652]
[2026-05-14 01:06:05,887: INFO: 293278742: Epoch [2/2] Train Loss: 0.6357 | Val Loss: 0.6474 | Val Acc: 0.6667]
[2026-05-14 01:06:05,934: INFO: 293278742: Best model saved with accuracy: 0.6667]
[2026-05-14 01:06:05,935: INFO: 293278742: Training stage completed successfully]
